# MQTT / HTTP / WebSocket Communication · Edge-Computing Notebook

Edge systems are made of parts that talk to each other: sensors, processors, dashboards, and the cloud. **How** they talk matters. In this lab you will move the same sensor reading across three communication protocols and see why each one exists.

```text
HTTP       -> request / response           (a client asks, the server answers)
MQTT       -> publish / subscribe          (devices publish to a broker, many subscribe)
WebSocket  -> persistent two-way stream    (server pushes live updates to clients)
```

You will:

- run a **Flask HTTP server** and hit it with GET and POST requests
- run a **Mosquitto MQTT broker** in Docker, with a publisher and a wildcard subscriber
- run a **WebSocket server** that pushes live readings to a browser page
- compare the three protocols side by side

The main idea is there is no single "best" protocol. Edge architectures pick the protocol that fits each link · and often use all three in one system.

## How this notebook works · cells and a Jupyter terminal

Steps that run once and return (writing a file, a `curl` request, starting a detached broker) are ordinary **notebook cells** · just run them here.

Each protocol also has a **server side that runs forever**: the Flask server, the MQTT publisher and subscriber, the WebSocket server. A notebook cell cannot run those · it would hang the kernel. For each one this notebook **writes a small script into your lab folder and asks you to run it in a Jupyter terminal.** Part 4 needs **two terminals side by side** (subscriber + publisher).

Two labels are used throughout:

- **[Notebook cell]** · run the cell right here.
- **[Terminal]** · open a Jupyter terminal (**File ▸ New ▸ Terminal**, or the Terminal tile in the Launcher) and run the given command there. You can open several terminals at once.

### One config, everywhere

Terminal shells do not share the notebook kernel's variables. So the setup cell writes `~/networkingLab/labEnv.sh` with your three ports, and every helper script starts with `source ~/networkingLab/labEnv.sh` · the notebook and every terminal always agree on `HTTP_PORT`, `MQTT_PORT`, and `WS_PORT` with no manual re-export.

**Requirements:** a Python 3 kernel on the Jetson Thor, Docker available to your user (for the Mosquitto broker).

In [ ]:
# Load the shared lab toolkit (labHelpers.py ships in the course repo next to
# this notebook). It provides pretty output, preflight checks, and checkpoints.
import sys, pathlib
searchDirs = [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3],
              pathlib.Path.home() / "EdgeClassHandson"]
helperDir = next((d for d in searchDirs if (d / "labHelpers.py").exists()), None)
assert helperDir is not None, "labHelpers.py not found - keep it next to this notebook"
sys.path.insert(0, str(helperDir))
from labHelpers import *

---
## Part 0 · Before You Begin

You are working on a shared **Jetson Thor** edge device · and this notebook runs there too. Your JupyterLab session lives on the Thor, so **every cell in this exercise runs on the Thor**, not on your own computer.

Because the Thor is shared:

- use your own home folder and keep your files under it
- prefix Docker resources and other shared names with your account name using `${USER}` (the cells below already do)
- use only the ports assigned to you · the setup cell derives three unique ones from your username

---
## Part 1 · Set Your Assigned Ports

You will run three small servers, so you need three **unique** ports · one each for HTTP, MQTT, and WebSocket. The cell below derives them from the digits in your username (`student07` gets `6007` / `7007` / `8007`), so nobody on the shared device collides. If your instructor assigned specific ports instead, force them with `portOverrides`.

**[Notebook cell]** This also creates `~/networkingLab` and writes `labEnv.sh` · the file every terminal helper script sources, so you never re-export ports by hand.

In [ ]:
import os

# Ports derive from your username: student07 -> HTTP 6007, MQTT 7007, WS 8007.
# Instructor-assigned ports? Use e.g.:
#   portOverrides={"HTTP_PORT": 6042, "MQTT_PORT": 7042, "WS_PORT": 8042}
labConfig = setupLab(
    "networkingLab",
    ports={"HTTP_PORT": 6000, "MQTT_PORT": 7000, "WS_PORT": 8000},
)

### Preflight · check your environment

Run this once at the start and fix any failing check before continuing.

In [ ]:
import os

preflight(
    [
        check("docker daemon reachable", dockerDaemonUp(),
              hint="The MQTT broker (Part 4) runs in Docker. Try <code>docker info</code> in a "
                   "Terminal; if it fails, ask your instructor.",
              link="https://docs.docker.com/engine/daemon/troubleshoot/",
              linkText="Docker daemon troubleshooting"),
        check("curl available", commandOnPath("curl"),
              hint="Part 3's HTTP client cells use curl. On the Thor it should already be "
                   "installed; ask your instructor if not.",
              link="https://curl.se/docs/manpage.html", linkText="curl manual"),
        check("python3 available", commandOnPath("python3"),
              hint="The venv cell below needs python3 on the PATH.",
              link="https://docs.python.org/3/library/venv.html", linkText="Python venv docs"),
    ],
    infoRows=[("your USER", os.environ.get("USER", "?")),
              ("your HTTP_PORT", os.environ.get("HTTP_PORT", "?")),
              ("your MQTT_PORT", os.environ.get("MQTT_PORT", "?")),
              ("your WS_PORT", os.environ.get("WS_PORT", "?"))],
)

**[Notebook cell]** This lab uses two small Python packages. Create a virtual environment for them (safe to re-run):

> This lab pins `paho-mqtt` below version 2.0 so the simple `mqtt.Client()` calls below work as written. Version 2.0 changed the client constructor and callback signatures; pinning keeps the examples consistent.

In [ ]:
%cd ~/networkingLab
!python3 -m venv venv
!venv/bin/pip install --quiet "paho-mqtt<2.0" websockets

### Checkpoint · Part 1

In [ ]:
import os

checkpoint(
    "Part 1 - ports and environment ready",
    [
        check("virtual environment created", fileExists("~/networkingLab/venv/bin/python3"),
              hint="Re-run the venv cell in Part 1 (<code>python3 -m venv venv</code>).",
              link="https://docs.python.org/3/library/venv.html", linkText="Python venv docs"),
        check("paho-mqtt installed (below 2.0)",
              commandSucceeds("bash -c '~/networkingLab/venv/bin/python3 -c \"import paho.mqtt.client\"'"),
              hint="Re-run the pip install line in Part 1.",
              link="https://eclipse.dev/paho/", linkText="Eclipse Paho clients"),
        check("websockets installed",
              commandSucceeds("bash -c '~/networkingLab/venv/bin/python3 -c \"import websockets\"'"),
              hint="Re-run the pip install line in Part 1.",
              link="https://websockets.readthedocs.io/", linkText="websockets library docs"),
        check("three unique ports exported", envVarSet("WS_PORT"),
              hint="Re-run the setupLab cell in Part 1 - it exports HTTP_PORT, MQTT_PORT, and "
                   "WS_PORT and writes labEnv.sh.",
              link="https://docs.python.org/3/library/os.html#os.environ", linkText="os.environ"),
    ],
    successNote="Environment ready - three ports, one venv, one labEnv.sh.",
    docLink="https://docs.python.org/3/library/venv.html",
    docLinkText="Python venv docs",
)

---
## Part 2 · How the Three Protocols Differ

```text
                HTTP                MQTT                  WebSocket
Pattern         request/response    publish/subscribe     persistent duplex
Who starts      client asks         broker routes         either side sends
Connection      open, ask, close    stay connected        stay connected
Best for        APIs, commands      many devices -> many  live dashboards, push
                                    consumers
Needs a broker  no                  yes (e.g. Mosquitto)  no
```

Quick intuition for the edge:

```text
HTTP       -> "give me the latest reading" or "here is one reading" (simple, universal)
MQTT       -> 100 sensors publish; a dashboard, a logger, and an alerter all subscribe
WebSocket  -> a browser dashboard that updates the instant new data arrives
```

You will build one of each, sending the same reading.

---
## Part 3 · HTTP Request and Response

HTTP is the protocol you already used in an earlier lab (you called InfluxDB's HTTP API with `curl` in the Grafana & Time-Series Database lab), so this part is deliberately short: just the core request/response pattern on its own.

**[Notebook cell]** Create `httpServer.py`:

In [ ]:
%%writefile httpServer.py
import os
import random
from datetime import datetime, timezone
from flask import Flask, jsonify, request

app = Flask(__name__)


@app.route("/reading", methods=["GET"])
def getReading():
    # the client ASKS, the server ANSWERS (request/response)
    return jsonify({
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "temperature": round(random.uniform(22, 40), 2),
    })


@app.route("/reading", methods=["POST"])
def postReading():
    # the client SENDS one reading, the server acknowledges
    return jsonify({"stored": request.json})


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=int(os.environ.get("HTTP_PORT", "6001")))

In [ ]:
# Preview the HTTP server we just wrote.
showFile("~/networkingLab/httpServer.py", language="python")

**[Notebook cell]** Install Flask into the venv and write the server's runner script:

In [ ]:
!venv/bin/pip install --quiet flask

In [ ]:
%%writefile runHTTPServer.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/networkingLab/labEnv.sh
cd ~/networkingLab
source venv/bin/activate
echo "HTTP server starting on port ${HTTP_PORT}. Press Ctrl-C to stop."
python3 httpServer.py

In [ ]:
!chmod +x runHTTPServer.sh

**[Terminal]** The server runs until you stop it, so start it in a Jupyter terminal and leave it running:

```bash
~/networkingLab/runHTTPServer.sh
```

**[Notebook cell]** Back here, make a GET and a POST request (the notebook plays the client, the terminal plays the server):

In [ ]:
!curl -s http://$DEVICE_ADDR:$HTTP_PORT/reading && echo

In [ ]:
!curl -s -X POST http://$DEVICE_ADDR:$HTTP_PORT/reading \
  -H "Content-Type: application/json" \
  -d '{"timestamp":"now","temperature":31.5}' && echo

**You should see:** the GET returns a fresh `timestamp`/`temperature` JSON object, and the POST echoes your reading back inside `{"stored": ...}`. In the server terminal, each request logs one line.

```text
Strength: simple, universal, works everywhere
Weakness: the client must keep asking ("polling") to notice new data
```

Run the checkpoint below **while the server is still running**, then stop it with **Ctrl-C** in its terminal.

### Checkpoint · Part 3

In [ ]:
import os
httpPort = os.environ.get("HTTP_PORT", "6001")

checkpoint(
    "Part 3 - HTTP request/response",
    [
        check("httpServer.py exists with both routes",
              fileContains("~/networkingLab/httpServer.py", "methods=[\"POST\"]"),
              hint="Re-run the %%writefile httpServer.py cell in Part 3.",
              link="https://flask.palletsprojects.com/en/stable/quickstart/",
              linkText="Flask quickstart"),
        check("server is listening on your HTTP_PORT",
              portListening(int(httpPort)),
              hint="Start <code>~/networkingLab/runHTTPServer.sh</code> in a Terminal and keep "
                   "it running while this checkpoint runs.",
              link="https://developer.mozilla.org/en-US/docs/Web/HTTP/Guides/Overview",
              linkText="MDN - HTTP overview"),
        check("GET /reading answers with a temperature",
              httpOk(f"http://{deviceAddress()}:{httpPort}/reading", expectText="temperature"),
              hint="With the server running in a Terminal, re-run the GET curl cell to see the "
                   "error, then this checkpoint.",
              link="https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Methods/GET",
              linkText="MDN - GET method"),
        check("POST /reading acknowledges with 'stored'",
              outputContains(f"curl -s -X POST http://{deviceAddress()}:{httpPort}/reading "
                             "-H 'Content-Type: application/json' -d '{\"temperature\": 31.5}'",
                             "stored"),
              hint="The POST route must return jsonify({\"stored\": request.json}) - re-run the "
                   "%%writefile cell and restart the server in its Terminal.",
              link="https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Methods/POST",
              linkText="MDN - POST method"),
    ],
    successNote="Request/response works both directions. You can stop the HTTP server (Ctrl-C) now.",
    docLink="https://developer.mozilla.org/en-US/docs/Web/HTTP/Guides/Overview",
    docLinkText="MDN - HTTP overview",
)

---
## Part 4 · MQTT Publish and Subscribe

MQTT decouples senders from receivers using a **broker**. Devices **publish** to a topic; any number of clients **subscribe** to that topic. This is the classic IoT pattern.

**[Notebook cell]** Run a Mosquitto broker in Docker. First write a minimal config (anonymous access is fine for this lab only):

In [ ]:
!mkdir -p ~/networkingLab/mosquitto

In [ ]:
%%writefile mosquitto/mosquitto.conf
listener 1883
allow_anonymous true

**[Notebook cell]** Start the broker on your assigned MQTT port (detached; the `rm -f` makes the cell safe to re-run):

In [ ]:
!docker rm -f ${USER}-mqttBroker 2>/dev/null || true
!docker run -d \
  --name ${USER}-mqttBroker \
  -p ${MQTT_PORT}:1883 \
  -v ~/networkingLab/mosquitto/mosquitto.conf:/mosquitto/config/mosquitto.conf \
  eclipse-mosquitto

**[Notebook cell]** Check it is running:

In [ ]:
import os
userName = os.environ.get("USER", "student01")

dockerPs(namePattern=f"{userName}-mqttBroker")
dockerLogs(f"{userName}-mqttBroker", tail=10)

### Checkpoint · Part 4 · broker

In [ ]:
import os
userName = os.environ.get("USER", "student01")

checkpoint(
    "Part 4a - MQTT broker up",
    [
        check("mosquitto.conf allows this lab's anonymous access",
              fileContains("~/networkingLab/mosquitto/mosquitto.conf", "allow_anonymous true"),
              hint="Re-run the %%writefile mosquitto/mosquitto.conf cell in Part 4.",
              link="https://mosquitto.org/man/mosquitto-conf-5.html",
              linkText="mosquitto.conf manual"),
        check("broker container is running",
              containerRunning(f"{userName}-mqttBroker"),
              hint="Re-run the docker run cell in Part 4, then check "
                   "<code>docker logs</code> with the status cell above.",
              link="https://mosquitto.org/documentation/", linkText="Mosquitto documentation"),
        check("broker is listening on your MQTT_PORT",
              portListening(int(os.environ.get("MQTT_PORT", "7001"))),
              hint="The run cell publishes <code>-p ${MQTT_PORT}:1883</code>. Confirm MQTT_PORT "
                   "in the Part 1 env card and that no other service holds that port.",
              link="https://docs.docker.com/engine/network/#published-ports",
              linkText="Published ports"),
    ],
    successNote="The middleman is up - now give it publishers and subscribers.",
    docLink="https://mosquitto.org/documentation/",
    docLinkText="Mosquitto documentation",
)

**[Notebook cell]** Create a publisher (the sensor) and a subscriber (a logger / dashboard / alerter · anything that wants the data):

In [ ]:
%%writefile mqttPublisher.py
import os
import json
import time
import random
from datetime import datetime, timezone
import paho.mqtt.client as mqtt

port = int(os.environ.get("MQTT_PORT", "7001"))
deviceID = os.environ.get("DEVICE_ID", "sensor01")
topic = f"edge/{deviceID}/telemetry"

client = mqtt.Client()
client.connect(os.environ.get("DEVICE_ADDR", "localhost"), port, 60)
client.loop_start()
print(f"publishing to topic: {topic}", flush=True)

while True:
    reading = {
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "device_id": deviceID,
        "temperature": round(random.uniform(22, 40), 2),
    }
    client.publish(topic, json.dumps(reading))
    print(f"published: {reading}", flush=True)
    time.sleep(2)

In [ ]:
%%writefile mqttSubscriber.py
import os
import paho.mqtt.client as mqtt

port = int(os.environ.get("MQTT_PORT", "7001"))
topic = "edge/+/telemetry"  # the + wildcard matches every device id


def onConnect(client, userData, flags, rc):
    print(f"connected (rc={rc}), subscribing to {topic}", flush=True)
    client.subscribe(topic)


def onMessage(client, userData, message):
    print(f"{message.topic} -> {message.payload.decode()}", flush=True)


client = mqtt.Client()
client.on_connect = onConnect
client.on_message = onMessage
client.connect(os.environ.get("DEVICE_ADDR", "localhost"), port, 60)
client.loop_forever()

In [ ]:
# Preview both MQTT clients.
showFile("~/networkingLab/mqttPublisher.py", language="python")
showFile("~/networkingLab/mqttSubscriber.py", language="python")

**[Notebook cell]** Write the runner scripts (the publisher takes an optional device id argument):

In [ ]:
%%writefile runSubscriber.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/networkingLab/labEnv.sh
cd ~/networkingLab
source venv/bin/activate
echo "subscriber starting on port ${MQTT_PORT}. Press Ctrl-C to stop."
python3 mqttSubscriber.py

In [ ]:
%%writefile runPublisher.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/networkingLab/labEnv.sh
cd ~/networkingLab
source venv/bin/activate
export DEVICE_ID="${1:-sensor01}"
echo "publisher starting as ${DEVICE_ID} on port ${MQTT_PORT}. Press Ctrl-C to stop."
python3 mqttPublisher.py

In [ ]:
!chmod +x runSubscriber.sh runPublisher.sh

Both clients loop forever, so this step needs **two terminals side by side**.

**[Terminal 1]** Start the subscriber and leave it running:

```bash
~/networkingLab/runSubscriber.sh
```

**[Terminal 2]** Start the publisher:

```bash
~/networkingLab/runPublisher.sh sensor01
```

**You should see:** Terminal 2 prints `published: {...}` every 2 seconds, and Terminal 1 prints a matching `edge/sensor01/telemetry -> {...}` line for each · even though you never connected them directly. The broker routed the messages by topic.

> **New term · broker / topic / publish-subscribe:** the broker (Mosquitto) is a middleman. Publishers send messages to a named **topic**; subscribers ask the broker for topics they care about. Senders and receivers never talk directly, which is what lets one broker fan out to many devices.

> **If it doesn't work:** if the subscriber prints nothing, confirm the broker is running (the Part 4a checkpoint above) and that both scripts show the same `MQTT_PORT` (they both source `labEnv.sh`, so they should). A `ConnectionRefusedError` means the broker isn't up or you're on the wrong port.

The subscriber prints every reading the publisher sends · but neither knows about the other; they only know the broker and the topic. Open a **third terminal** (or reuse one) and start a **second** publisher with a different device id · the same subscriber receives both, thanks to the `+` wildcard:

**[Terminal 3]**

```bash
~/networkingLab/runPublisher.sh sensor02
```

Keep at least one publisher running for the checkpoint below · it listens on the topic for a few seconds and expects real traffic. Afterwards, stop the publishers and subscriber with **Ctrl-C**.

### Checkpoint · Part 4 · pub/sub

**[Notebook cell]** First write a tiny probe the checkpoint uses: it subscribes to `edge/+/telemetry` for up to 8 seconds and succeeds if any message arrives.

In [ ]:
%%writefile mqttProbe.py
import os
import sys
import time
import paho.mqtt.client as mqtt

port = int(os.environ.get("MQTT_PORT", "7001"))
received = []


def onMessage(client, userData, message):
    received.append(message)
    print(f"{message.topic} -> {message.payload.decode()}", flush=True)


client = mqtt.Client()
client.on_message = onMessage
client.connect(os.environ.get("DEVICE_ADDR", "localhost"), port, 60)
client.subscribe("edge/+/telemetry")
client.loop_start()
deadline = time.time() + 8
while time.time() < deadline and not received:
    time.sleep(0.2)
client.loop_stop()
sys.exit(0 if received else 1)

In [ ]:
checkpoint(
    "Part 4b - MQTT pub/sub round trip",
    [
        check("publisher and subscriber written",
              fileContains("~/networkingLab/mqttSubscriber.py", "edge/+/telemetry"),
              hint="Re-run the %%writefile mqttPublisher.py / mqttSubscriber.py cells in Part 4.",
              link="https://mqtt.org/", linkText="MQTT.org"),
        check("runner scripts written",
              fileExists("~/networkingLab/runPublisher.sh"),
              hint="Re-run the %%writefile runSubscriber.sh / runPublisher.sh and chmod cells.",
              link="https://www.gnu.org/software/bash/manual/bash.html",
              linkText="Bash reference manual"),
        check("a message arrived on edge/+/telemetry (takes up to ~8s)",
              commandSucceeds("bash -c '~/networkingLab/venv/bin/python3 ~/networkingLab/mqttProbe.py'",
                              timeoutSeconds=20),
              hint="Keep <code>runPublisher.sh</code> running in a Terminal while this checkpoint "
                   "runs - the probe subscribes for 8 seconds and needs live traffic.",
              link="https://mosquitto.org/documentation/", linkText="Mosquitto documentation"),
    ],
    successNote="Publish/subscribe verified end to end - the broker fans your readings out to "
                "anyone who subscribes.",
    docLink="https://mqtt.org/",
    docLinkText="MQTT.org",
)

```text
Strength: many-to-many, devices don't need to know each other, scales to large fleets
Weakness: needs a broker, slightly more to set up than plain HTTP
```

This pub/sub pattern is exactly what the Fleet Management lab uses to monitor many devices at once.

---
## Part 5 · WebSocket Live Server Push

HTTP makes the client ask repeatedly. A **WebSocket** keeps one connection open so the server can **push** data the moment it changes · ideal for a live dashboard.

**[Notebook cell]** Create the WebSocket server and its runner:

In [ ]:
%%writefile wsServer.py
import os
import json
import asyncio
import random
from datetime import datetime, timezone
import websockets

port = int(os.environ.get("WS_PORT", "8001"))


async def stream(websocket):
    print("client connected", flush=True)
    try:
        while True:
            reading = {
                "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "temperature": round(random.uniform(22, 40), 2),
            }
            await websocket.send(json.dumps(reading))
            await asyncio.sleep(1)
    except websockets.ConnectionClosed:
        print("client disconnected", flush=True)


async def main():
    async with websockets.serve(stream, "0.0.0.0", port):
        print(f"websocket server on port {port}", flush=True)
        await asyncio.Future()  # run forever


if __name__ == "__main__":
    asyncio.run(main())

In [ ]:
%%writefile runWSServer.sh
#!/usr/bin/env bash
set -uo pipefail
source ~/networkingLab/labEnv.sh
cd ~/networkingLab
source venv/bin/activate
echo "WebSocket server starting on port ${WS_PORT}. Press Ctrl-C to stop."
python3 wsServer.py

In [ ]:
!chmod +x runWSServer.sh

**[Terminal]** Start the WebSocket server and leave it running:

```bash
~/networkingLab/runWSServer.sh
```

**[Notebook cell]** Before the browser, prove the push works from right here: this tiny client connects once and receives three pushed readings (one per second, no re-requests):

In [ ]:
%%writefile wsProbe.py
import os
import json
import asyncio
import websockets


async def main():
    url = f"ws://{os.environ.get('DEVICE_ADDR', 'localhost')}:{os.environ.get('WS_PORT', '8001')}"
    async with websockets.connect(url) as connection:
        for _ in range(3):
            message = await asyncio.wait_for(connection.recv(), timeout=5)
            print(f"pushed by server: {message}", flush=True)

asyncio.run(main())

In [ ]:
!venv/bin/python3 wsProbe.py

**You should see:** three readings arrive ~1 second apart · the server pushed each one over the single open connection. The server terminal prints `client connected` and then `client disconnected`.

**[Notebook cell]** Now create the browser client, `wsClient.html`:

In [ ]:
%%writefile wsClient.html
<!doctype html>
<html>
  <head><title>Live Edge Stream</title></head>
  <body>
    <h1>Live Edge Stream (WebSocket)</h1>
    <pre id="out">connecting...</pre>
    <script>
      const ws = new WebSocket("ws://THOR_ADDRESS:WS_PORT");
      ws.onmessage = (event) => {
        document.getElementById("out").textContent = event.data;
      };
      ws.onclose = () => {
        document.getElementById("out").textContent = "connection closed";
      };
    </script>
  </body>
</html>

In [ ]:
# Fill in your WS_PORT automatically; THOR_ADDRESS you must still replace by hand.
import os
import pathlib

htmlPath = pathlib.Path.home() / "networkingLab" / "wsClient.html"
htmlPath.write_text(htmlPath.read_text().replace("WS_PORT", os.environ.get("WS_PORT", "8001")))
showFile(htmlPath, language="html")
showNote("The file now uses your WebSocket port. Still replace <code>THOR_ADDRESS</code> with "
         "the Thor's IP address before opening the page on your own computer.", kind="tip")

**[On your own computer]** Copy the HTML file over and open it in a browser:

```text
scp thor:~/networkingLab/wsClient.html .
```

Open `wsClient.html` from your computer.

**You should see:** the line on the page change once per second on its own, with no page reload. The server terminal prints `client connected` when the browser attaches. If the page stays stuck on `connecting...`, you didn't replace `THOR_ADDRESS` with a real value, or the port isn't reachable from your laptop.

The reading on the page updates every second with no page reload and no repeated requests: the server pushed each update over the open connection.

### Checkpoint · Part 5

Run this **while the WebSocket server is still running**, then stop the server with **Ctrl-C** in its terminal.

In [ ]:
import os

checkpoint(
    "Part 5 - WebSocket live push",
    [
        check("wsServer.py exists and serves forever",
              fileContains("~/networkingLab/wsServer.py", "websockets.serve"),
              hint="Re-run the %%writefile wsServer.py cell in Part 5.",
              link="https://websockets.readthedocs.io/", linkText="websockets library docs"),
        check("server is listening on your WS_PORT",
              portListening(int(os.environ.get("WS_PORT", "8001"))),
              hint="Start <code>~/networkingLab/runWSServer.sh</code> in a Terminal and keep it "
                   "running while this checkpoint runs.",
              link="https://developer.mozilla.org/en-US/docs/Web/API/WebSockets_API",
              linkText="MDN - WebSockets API"),
        check("server pushes readings over one connection",
              commandSucceeds("bash -c '~/networkingLab/venv/bin/python3 ~/networkingLab/wsProbe.py'",
                              timeoutSeconds=25),
              hint="With the server running, re-run the wsProbe cell to see the error, then this "
                   "checkpoint.",
              link="https://developer.mozilla.org/en-US/docs/Web/API/WebSocket",
              linkText="MDN - WebSocket interface"),
        check("browser client written with a WebSocket",
              fileContains("~/networkingLab/wsClient.html", "new WebSocket"),
              hint="Re-run the %%writefile wsClient.html cell in Part 5.",
              link="https://developer.mozilla.org/en-US/docs/Web/API/WebSocket/message_event",
              linkText="MDN - WebSocket message event"),
    ],
    successNote="Live server push verified - one connection, updates the instant they exist.",
    docLink="https://developer.mozilla.org/en-US/docs/Web/API/WebSockets_API",
    docLinkText="MDN - WebSockets API",
)

```text
Strength: instant push, low overhead per update, true two-way communication
Weakness: a long-lived connection per client; more state for the server to hold
```

---
## Part 6 · Compare the Three

You sent the same reading three ways. Put the differences side by side:

| Question | HTTP | MQTT | WebSocket |
|---|---|---|---|
| How does a client get new data? | it asks | broker pushes | server pushes |
| Good for one-off command? | yes | overkill | overkill |
| Good for 100 devices -> 3 apps? | awkward | ideal | awkward |
| Good for a live browser view? | polling | needs a bridge | ideal |
| Connection cost | low (brief) | one to broker | one per client |

Real edge systems mix them. A common pattern:

```text
Sensors --MQTT--> Broker --MQTT--> Edge Processor
Edge Processor --HTTP--> Cloud API (store / aggregate)
Edge Processor --WebSocket--> Live local dashboard
```

Each protocol is used where its strengths fit.

---
## Cleanup

**[Notebook cell]** Stop the MQTT broker and remove it (safe to run any number of times). Any servers still running in terminals stop with **Ctrl-C** there.

In [ ]:
!docker rm -f ${USER}-mqttBroker 2>/dev/null || true

**[Notebook cell]** Optional · remove your lab folder, venv included (uncomment to run). Only remove your own files and containers.

In [ ]:
# !rm -rf ~/networkingLab

### Lab scorecard

Run this any time for a live summary of every checkpoint in this kernel session. If a row is incomplete, scroll up, fix that part, re-run its checkpoint, and re-run this cell.

In [ ]:
labSummary("MQTT / HTTP / WebSocket Communication")

---
## Connect to the Rest of the Course

You now have the communication toolkit edge systems are built from:

```text
HTTP       -> request/response APIs and cloud calls          (used with curl in the TSDB lab)
MQTT       -> many-to-many device telemetry                  (used next, for the fleet)
WebSocket  -> live push to dashboards
```

A real project chooses per link: MQTT from a fleet of sensors to a broker, HTTP to a cloud backend, WebSocket to operator dashboards. The main idea: communication protocol is an architecture decision, and matching the protocol to the traffic pattern is what keeps an edge system responsive and scalable.

---
## Key Terms

- **HTTP:** request/response · a client asks, the server answers, then the connection closes. Used for APIs and cloud calls.
- **Polling:** repeatedly sending HTTP requests to check for new data; simple but wasteful.
- **MQTT:** a lightweight publish/subscribe protocol for many-to-many messaging through a broker.
- **Broker:** the MQTT middleman (here, Mosquitto) that receives published messages and delivers them to subscribers.
- **Topic:** a named channel messages are published to and subscribed from; `+` is a single-level wildcard (`edge/+/telemetry` matches every device).
- **Publish / subscribe:** publishers send to a topic without knowing who listens; subscribers receive from a topic without knowing who sent it.
- **WebSocket:** a single long-lived connection that lets the server *push* data to the client the moment it changes · ideal for live dashboards.